# Notebook 11 — Build RAG Evidence Retrieval

## Goal
Retrieve contemporaneous news evidence for each forecast. This is where RAG begins.

---

## IMPORTANT: What RAG does and does NOT do

**RAG does NOT predict.** The XGBoost models already produced predictions in Notebooks 08–09.

**RAG retrieves news evidence** to *explain* each existing forecast.
For each forecast month `t`, we find articles published on or before `forecast_date = t`
that are most relevant to the labor market. These articles become the evidence base
that Notebook 12 passes to DeepSeek for explanation generation.

---

## Hard rule: strict time filter

```
For each forecast at time t:  article_date <= forecast_date
```

No exceptions. An article published after the forecast date cannot have influenced the forecast.
Citing future articles would make explanations misleading.

---

## What can go wrong
- If `article_date > forecast_date` appears in the evidence, it is a leakage bug
- Retrieval may return few results for early months (thin GDELT coverage before 2014)
- All 0-evidence months must be documented and explained

---

## What you must inspect
- Example evidence blocks: do the retrieved articles match the forecast month?
- Are all article dates before the corresponding forecast date?
- Are retrieved articles actually about the US labor market?

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

CLEAN_NEWS_PATH = DRIVE_ROOT / 'data' / 'interim' / 'news' / 'clean_news_approved.parquet'
PREDICTIONS_PATH = DRIVE_ROOT / 'outputs' / 'predictions' / 'predictions_macro_news.parquet'
EXPLANATIONS_DIR = DRIVE_ROOT / 'outputs' / 'explanations'

ap_path = DRIVE_ROOT / 'approvals' / '09_xgboost_models_approved.json'
if not ap_path.exists():
    raise FileNotFoundError('STOP: Notebook 09 not approved.')
with open(ap_path) as f:
    ap = json.load(f)
assert ap['approved'], 'NB 09 not approved.'

with open(REPO_DIR / 'configs' / 'evidence_config.yaml') as f:
    ev_cfg = yaml.safe_load(f)

MAX_ARTICLES = ev_cfg['retrieval']['max_articles_per_forecast']
PREFERRED_WINDOW = ev_cfg['retrieval']['preferred_window_months']

news = pd.read_parquet(CLEAN_NEWS_PATH)
news['seendate'] = pd.to_datetime(news['seendate'], errors='coerce')
news = news.dropna(subset=['seendate', 'title', 'url'])

preds = pd.read_parquet(PREDICTIONS_PATH)
preds['forecast_date'] = pd.to_datetime(preds['forecast_date'])
preds = preds.sort_values('forecast_date').reset_index(drop=True)

print(f'Clean news: {news.shape}')
print(f'Predictions to explain: {len(preds)} forecasts')
print(f'Forecast date range: {preds["forecast_date"].min().date()} to {preds["forecast_date"].max().date()}')

## Retrieval function

In [ ]:
QUERY_GROUP_WEIGHTS = ev_cfg['retrieval']['query_group_weights']

def retrieve_evidence(news_df, forecast_date, max_articles=10, preferred_window_months=2):
    """
    Retrieve news articles published on or before forecast_date.
    Prefer articles from the preferred_window_months preceding forecast_date.
    Applies strict time filter: article_date <= forecast_date.
    """
    window_start = forecast_date - pd.DateOffset(months=preferred_window_months)

    # Strict filter: no future articles
    eligible = news_df[news_df['seendate'] <= forecast_date].copy()

    # Prefer articles from preferred window
    preferred = eligible[eligible['seendate'] >= window_start].copy()
    fallback = eligible[eligible['seendate'] < window_start].copy()

    # Apply query group weights (higher weight = include more articles from that group)
    def apply_weights(df):
        if len(df) == 0:
            return df
        df = df.copy()
        df['weight'] = df['query_group'].map(QUERY_GROUP_WEIGHTS).fillna(1.0)
        return df

    preferred = apply_weights(preferred)
    fallback = apply_weights(fallback)

    # Sort by weight then recency
    preferred = preferred.sort_values(['weight', 'seendate'], ascending=[False, False])
    fallback = fallback.sort_values(['weight', 'seendate'], ascending=[False, False])

    # Combine: fill up to max_articles from preferred, then fallback
    result = pd.concat([preferred.head(max_articles), fallback]).head(max_articles)

    result['forecast_date'] = forecast_date
    result['article_date'] = result['seendate']

    return result[['forecast_date', 'article_date', 'url', 'title', 'query_group',
                   'domain', 'text_for_nlp']].copy()

print('Retrieval function defined.')

## Assertion function

In [ ]:
def assert_evidence_time_filter(evidence_df, name='evidence'):
    """
    Verify that no retrieved article is dated after the corresponding forecast_date.
    This is the critical anti-leakage check for RAG.
    """
    bad = evidence_df[evidence_df['article_date'] > evidence_df['forecast_date']]
    assert len(bad) == 0, (
        f'RAG TIME LEAKAGE: {len(bad)} articles are dated AFTER forecast_date!\n'
        f'{bad[["forecast_date", "article_date", "title"]].head(5).to_string()}'
    )
    print(f'  [{name}] Time filter assertion passed: all {len(evidence_df)} articles are dated <= forecast_date.')

print('Assertion function defined.')

## Run retrieval for all forecast months

In [ ]:
from tqdm import tqdm

all_evidence = []
retrieval_stats = []

for _, row in tqdm(preds.iterrows(), total=len(preds), desc='Retrieving evidence'):
    fd = row['forecast_date']
    ev = retrieve_evidence(news, fd, max_articles=MAX_ARTICLES, preferred_window_months=PREFERRED_WINDOW)
    all_evidence.append(ev)
    retrieval_stats.append({
        'forecast_date': fd,
        'prediction': row['prediction'],
        'actual': row['actual'],
        'n_articles_retrieved': len(ev),
        'query_groups_found': sorted(ev['query_group'].unique().tolist()) if len(ev) > 0 else [],
    })

evidence_df = pd.concat(all_evidence, ignore_index=True) if all_evidence else pd.DataFrame()
stats_df = pd.DataFrame(retrieval_stats)

print(f'Total evidence rows: {len(evidence_df)}')
print(f'Forecasts with 0 articles: {(stats_df["n_articles_retrieved"] == 0).sum()}')
print(f'Mean articles per forecast: {stats_df["n_articles_retrieved"].mean():.1f}')

## Diagnostic inspection

In [ ]:
print('=== Retrieval Stats Summary ===')
print(stats_df.describe().to_string())
print()

print('=== Months with 0 retrieved articles ===')
zero_months = stats_df[stats_df['n_articles_retrieved'] == 0]
if len(zero_months) > 0:
    print(zero_months[['forecast_date', 'prediction']].to_string(index=False))
    print('WARNING: 0-evidence months will receive template explanations in NB 12.')
else:
    print('  No months with 0 articles.')

In [ ]:
def show_evidence_block(evidence_df, forecast_date, preds_df):
    fd = pd.Timestamp(forecast_date)
    ev = evidence_df[evidence_df['forecast_date'] == fd]
    pred_row = preds_df[preds_df['forecast_date'] == fd]
    pred = pred_row['prediction'].iloc[0] if len(pred_row) > 0 else np.nan
    actual = pred_row['actual'].iloc[0] if len(pred_row) > 0 else np.nan

    print(f'\n{"="*60}')
    print(f'Forecast date: {fd.date()}')
    print(f'Predicted PAYEMS change: {pred:+.0f}K | Actual: {actual:+.0f}K')
    print(f'Retrieved articles ({len(ev)}):')
    for i, row in ev.iterrows():
        print(f'  [{row["query_group"]:20s}] {row["article_date"].date()} — {row["title"][:80]}')
        print(f'                            URL: {row["url"][:70]}')

# Show examples for the most recent 3 forecast months
for fd in preds['forecast_date'].nlargest(3).values:
    show_evidence_block(evidence_df, fd, preds)

## Run the critical time-filter assertion

In [ ]:
print('Running critical time-filter assertion...')
if len(evidence_df) > 0:
    assert_evidence_time_filter(evidence_df)
else:
    print('WARNING: No evidence retrieved. Check news data and date ranges.')

# Additional checks
assert 'forecast_date' in evidence_df.columns, 'Missing forecast_date column'
assert 'article_date' in evidence_df.columns, 'Missing article_date column'
assert 'title' in evidence_df.columns, 'Missing title column'
assert 'url' in evidence_df.columns, 'Missing url column'
assert 'query_group' in evidence_df.columns, 'Missing query_group column'
print('  Schema assertions passed.')
print('\nAll assertions passed.')

## Manual Approval Gate

**Before approving:**
1. Time-filter assertion passed (no future articles in evidence)
2. Example evidence blocks show relevant articles
3. Retrieved article dates are on or before the forecast date
4. No more than ~20% of forecasts have 0 retrieved articles
5. Query group distribution looks reasonable

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
evidence_path = EXPLANATIONS_DIR / 'retrieved_evidence.parquet'
evidence_df.to_parquet(evidence_path, index=False)
print(f'Evidence saved: {evidence_path}')

stats_path = DRIVE_ROOT / 'outputs' / 'data_quality' / 'retrieval_stats.csv'
stats_df.to_csv(stats_path, index=False)

approval = {
    'approved': True,
    'notebook': '11_build_rag_evidence_retrieval.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(CLEAN_NEWS_PATH), str(PREDICTIONS_PATH)],
    'output_artifacts': [str(evidence_path)],
    'row_counts': {
        'total_evidence_rows': int(len(evidence_df)),
        'forecasts_with_evidence': int((stats_df['n_articles_retrieved'] > 0).sum()),
        'forecasts_with_zero_evidence': int((stats_df['n_articles_retrieved'] == 0).sum()),
    },
    'warnings': ['time_filter_assertion_passed'],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '11_rag_retrieval_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 11 complete. Proceed to 12_generate_deepseek_explanations.ipynb')